# TTS Corpus Ingest

Reads a real speech corpus and writes a manifest of `(text, audio)` rows in the format that
[docs/manifest.md](../docs/manifest.md) defines. This is step 2b of the TTS dataset pipeline
described in [docs/pipeline.md](../docs/pipeline.md): an alternative to donor TTS synthesis
([`tts_dataset_gen.ipynb`](tts_dataset_gen.ipynb)) that takes its audio from an existing
recording instead of synthesising it. [docs/registry.md](../docs/registry.md) describes the
per-language registry that later synthesis and judging steps read; this notebook does not use it,
since it neither synthesises nor scores.

### Pipeline stages

| # | Stage | Description |
|---|---|---|
| 0 | Configuration | Set the corpus kind, its inputs, and the speaker and hours rules |
| 1 | Load corpus rows | Read `(id, text, speaker, audio)` from a Hugging Face dataset, an LJSpeech directory, or a CSV |
| 2 | Speaker accounting | A first pass sums audio duration and row count per speaker |
| 3 | Speaker selection | Keep an explicit speaker list, or the top N by hours |
| 4 | Text validation | Refuse rows with empty text, a bracketed annotation, or no letter |
| 5 | Write the manifest | A second pass decodes audio, asserts mono, applies the hours cap, and writes WAV files |
| 6 | Manifest, sidecar and summary | Write `manifest.jsonl`, its `.meta.json`, and print the counts |

### Inputs

- `CORPUS_KIND=hf`: a Hugging Face dataset, read in streaming mode.
- `CORPUS_KIND=ljspeech`: a local directory with `metadata.csv` (`id|text` or `id|text|normalized`)
  and a `wavs/` folder.
- `CORPUS_KIND=csv`: a local CSV file with configurable text and audio-path columns.
- Optional speaker selection (`SPEAKERS` or `TOP_SPEAKERS`) and an hours cap (`MAX_HOURS`), so a
  multi-speaker corpus yields single-speaker material.

### Outputs

- `OUTPUT_DIR/manifest.jsonl`: one row per kept clip, `source.kind` set to `corpus`, `speaker` set
  where the corpus names one.
- `OUTPUT_DIR/manifest.jsonl.meta.json`: the corpus id, its config and splits, the Hub revision
  read, the speaker rule, and the row counts.
- `OUTPUT_DIR/wavs/`: the audio, mono PCM WAV at each clip's own native sample rate.

### Hardware

CPU only. There is no GPU step, no ONNX model, and no quality score here — this notebook reads a
corpus, refuses bad text, and writes the manifest and audio; scoring is the measure notebook's
job. `CORPUS_KIND=hf` needs outbound internet access and can download up to a few hundred
megabytes, so it is not executed by CI (see the banner below).

> This notebook was written with AI assistance (Claude, via Claude Code). Verify outputs before relying on them.


In [ ]:
# Parameters
SMOKE = False   # True: skip CORPUS_KIND=hf and ingest a tiny built-in LJSpeech fixture, CPU, no network
N_SAMPLES = 4   # row count of the built-in SMOKE fixture (mixes rows that pass and fail text validation)


> ⚠ **`CORPUS_KIND=hf` needs outbound internet access** and a Hugging Face dataset download that can reach a few hundred megabytes; that path is not run by CI.
> `SMOKE=True` (see the parameters cell) never reaches the network: it builds a tiny built-in LJSpeech-style fixture and ingests that, so the smoke path is CPU-only and fast.


---
## NGI0 Grant Attribution

> Developed by TigreGotico for OpenVoiceOS.

> This project was funded through the NGI0 Commons Fund, a fund established by NLnet with financial support from the European Commission's Next Generation Internet programme, under the aegis of DG Communications Networks, Content and Technology under grant agreement No 101135429. Additional funding is made available by the Swiss State Secretariat for Education, Research and Innovation (SERI).

**NGI work item:** WP3 — TTS dataset pipeline, corpus ingest step
**CI:** executed on the smoke path (`SMOKE=True`, no network, no GPU)

---


---
## 0 · Configuration

All parameters are set as environment variables via `os.environ.setdefault()`. Override any of
them from your shell, a `.env` file, or `%env` magic **before** running the config cell.

The pipeline, the manifest format and the language registry are described in
[docs/pipeline.md](../docs/pipeline.md), [docs/manifest.md](../docs/manifest.md) and
[docs/registry.md](../docs/registry.md).

**Edit the values below to match your corpus, then run the cell.**


In [ ]:
import os


def _build_smoke_fixture(n_samples):
    """A tiny built-in LJSpeech-style fixture: n_samples rows mixing text that
    passes validation with text that Stage 4 refuses (bracketed, no letter), and
    silent PCM audio — ingest never reads what a clip says, only its text and
    its channel count, so silence exercises the same code paths as real speech.
    """
    import tempfile
    import numpy as np
    import soundfile as sf

    texts = [
        "Hello there, this is a valid sentence.",
        "Another perfectly fine line of text.",
        "[inaudible]",   # refused: bracketed annotation
        "12345",         # refused: no letter
    ]
    n = max(int(n_samples), 1)
    texts = (texts * ((n // len(texts)) + 1))[:n]

    fixture_dir = Path(tempfile.mkdtemp(prefix="tts_corpus_ingest_smoke_"))
    (fixture_dir / "wavs").mkdir()
    with open(fixture_dir / "metadata.csv", "w", encoding="utf-8") as f:
        for i, text in enumerate(texts):
            row_id = f"smoke{i + 1:03d}"
            sf.write(str(fixture_dir / "wavs" / f"{row_id}.wav"),
                     np.zeros(8000, dtype="float32"), 16000, subtype="PCM_16")
            f.write(f"{row_id}|{text}\n")
    return str(fixture_dir)

if SMOKE:
    from pathlib import Path
    os.environ["CORPUS_KIND"] = "ljspeech"
    os.environ["LJSPEECH_DIR"] = _build_smoke_fixture(N_SAMPLES)
    os.environ["LJSPEECH_SPEAKER"] = "smoke"
    print(f"SMOKE=True: ingesting the built-in fixture at {os.environ['LJSPEECH_DIR']}")




def _default_output_dir():
    """Somewhere writable that survives the session on the current platform."""
    if os.path.isdir("/kaggle"):
        return "/kaggle/working/corpus_ingest_out"
    if os.path.isdir("/content"):
        return "/content/corpus_ingest_out"
    return "./corpus_ingest_out"


# ──────────────────────────────────────────────
# GENERAL
# ──────────────────────────────────────────────
os.environ.setdefault("CORPUS_KIND",   "hf")                    # hf | ljspeech | csv
os.environ.setdefault("OUTPUT_DIR",    _default_output_dir())   # manifest + wavs/ are written here
os.environ.setdefault("CORPUS_LANG",   "en")                    # BCP-47 tag with the variety, e.g. pt-PT
os.environ.setdefault("CORPUS_NAME",   "")                      # source.name; defaults per kind below
os.environ.setdefault("CORPUS_URL",    "")                      # source.url; defaults to the HF dataset page for CORPUS_KIND=hf
os.environ.setdefault("MANIFEST_NAME", "manifest.jsonl")        # manifest filename inside OUTPUT_DIR

# Hugging Face Hub cache. Never HF_HOME: that also moves the auth token, and an unauthenticated
# Hub call fails on a gated or private dataset. Left unset here, the Hub uses its own default
# cache directory; set HF_HUB_CACHE yourself before running this cell to point at a disk with room.

# ──────────────────────────────────────────────
# CORPUS_KIND=hf — a Hugging Face dataset, read in streaming mode
# ──────────────────────────────────────────────
os.environ.setdefault("HF_DATASET",      "")                    # repo id, e.g. "org/dataset-name"
os.environ.setdefault("HF_CONFIG",       "")                    # dataset config name, empty = the default config
os.environ.setdefault("HF_SPLITS",       "train")               # comma list, e.g. "train,validation"
os.environ.setdefault("HF_REVISION",     "")                    # commit sha or tag, empty = the current default branch
os.environ.setdefault("TEXT_FIELD",      "text")                # column holding the transcript
os.environ.setdefault("AUDIO_FIELD",     "audio")                # column holding the audio
os.environ.setdefault("SPEAKER_FIELD",   "")                    # column holding the speaker id, empty = none
# Streaming pulls one shard at a time; a short sleep every HF_PACE_EVERY rows keeps the
# duration-scanning pass from becoming a gapless bulk fetch.
os.environ.setdefault("HF_PACE_EVERY",   "50")
os.environ.setdefault("HF_PACE_SECONDS", "1.0")

# ──────────────────────────────────────────────
# CORPUS_KIND=ljspeech — a local directory with metadata.csv (id|text or id|text|normalized) and wavs/
# ──────────────────────────────────────────────
os.environ.setdefault("LJSPEECH_DIR",      "")                  # directory holding metadata.csv and wavs/
os.environ.setdefault("LJSPEECH_SPEAKER",  "")                  # this directory's single speaker id, empty = none named

# ──────────────────────────────────────────────
# CORPUS_KIND=csv — a local CSV with configurable text and audio-path columns
# ──────────────────────────────────────────────
os.environ.setdefault("CSV_PATH",           "")                 # path to the CSV file
os.environ.setdefault("CSV_TEXT_COL",       "text")             # column holding the transcript
os.environ.setdefault("CSV_AUDIO_COL",      "audio")            # column holding the audio file path
os.environ.setdefault("CSV_AUDIO_BASE_DIR", "")                 # audio paths are relative to this, default = the CSV's directory
os.environ.setdefault("CSV_SPEAKER_COL",    "")                 # column holding the speaker id, empty = none
os.environ.setdefault("CSV_ID_COL",         "")                 # column holding a row id, empty = the row's position

# ──────────────────────────────────────────────
# SPEAKER SELECTION
#   A multi-speaker corpus is narrowed to single-speaker material. Set at most one of
#   SPEAKERS or TOP_SPEAKERS; leave both empty (or 0) to keep every speaker.
# ──────────────────────────────────────────────
os.environ.setdefault("SPEAKERS",      "")                      # comma list of speaker ids to keep
os.environ.setdefault("TOP_SPEAKERS",  "0")                     # keep the N speakers with the most audio, 0 = off
os.environ.setdefault("MAX_HOURS",     "")                      # cap on the hours written, empty = no cap


def _env(key, default=""):
    return os.environ.get(key, default)


def _flag(key):
    return _env(key, "0").strip().lower() in ("1", "true", "yes")


def _int_env(key, default=0):
    v = _env(key).strip()
    return int(v) if v else default


def _float_env(key, default=None):
    v = _env(key).strip()
    return float(v) if v else default


CORPUS_KIND = _env("CORPUS_KIND")
CORPUS_LANG = _env("CORPUS_LANG")
OUTPUT_DIR = _env("OUTPUT_DIR")

assert CORPUS_KIND in ("hf", "ljspeech", "csv"), f"CORPUS_KIND must be hf, ljspeech or csv, got {CORPUS_KIND!r}"

print("Configuration loaded.")
print(f"  Corpus kind    : {CORPUS_KIND}")
print(f"  Language       : {CORPUS_LANG}")
print(f"  Output dir     : {OUTPUT_DIR}")
if CORPUS_KIND == "hf":
    print(f"  HF dataset     : {_env('HF_DATASET')}  config={_env('HF_CONFIG') or '(default)'}  splits={_env('HF_SPLITS')}")
    print(f"  Fields         : text={_env('TEXT_FIELD')}  audio={_env('AUDIO_FIELD')}  speaker={_env('SPEAKER_FIELD') or '(none)'}")
elif CORPUS_KIND == "ljspeech":
    print(f"  LJSpeech dir   : {_env('LJSPEECH_DIR')}  speaker={_env('LJSPEECH_SPEAKER') or '(none)'}")
else:
    print(f"  CSV path       : {_env('CSV_PATH')}  text={_env('CSV_TEXT_COL')}  audio={_env('CSV_AUDIO_COL')}  speaker={_env('CSV_SPEAKER_COL') or '(none)'}")
print(f"  Speaker rule   : SPEAKERS={_env('SPEAKERS') or '(none)'}  TOP_SPEAKERS={_int_env('TOP_SPEAKERS')}  MAX_HOURS={_env('MAX_HOURS') or '(no cap)'}")


---
## 1 · Load corpus rows

Each row is normalised to the same shape regardless of `CORPUS_KIND`: an id, a text, an optional
speaker, and either a decoded audio array (`CORPUS_KIND=hf`) or a path to a local WAV
(`CORPUS_KIND=ljspeech` or `csv`). The `hf` reader pulls one Hub shard at a time in streaming mode
and paces itself; the other two readers read local files directly.

A Hugging Face row is keyed `<dataset>:<split>:<row index>` — the row's position inside its own
split, never a bare position across splits. An `ljspeech` or `csv` row is keyed
`<corpus name>:<row id>`.

| Variable | Meaning |
|---|---|
| `CORPUS_KIND` | `hf`, `ljspeech` or `csv` |
| `HF_DATASET`, `HF_CONFIG`, `HF_SPLITS`, `HF_REVISION` | Which Hugging Face dataset, config, splits and revision |
| `TEXT_FIELD`, `AUDIO_FIELD`, `SPEAKER_FIELD` | Column names for `CORPUS_KIND=hf` |
| `LJSPEECH_DIR`, `LJSPEECH_SPEAKER` | Directory and speaker id for `CORPUS_KIND=ljspeech` |
| `CSV_PATH`, `CSV_TEXT_COL`, `CSV_AUDIO_COL`, `CSV_AUDIO_BASE_DIR`, `CSV_SPEAKER_COL`, `CSV_ID_COL` | Columns for `CORPUS_KIND=csv` |
| `HF_HUB_CACHE`, `HF_PACE_EVERY`, `HF_PACE_SECONDS` | Hub cache location and download pacing |


In [ ]:
%pip install --quiet --pre "datasets>=5.0.1" "huggingface_hub>=0.25.0,<2.0" "soundfile>=0.14.0" "numpy>=2.5.3"
# huggingface_hub is capped below 2.0 here because datasets>=5.0.1 itself requires huggingface_hub<2.0;
# a higher floor on this line alone would make the install unresolvable.


In [ ]:
import csv
import json
import re
import time
from pathlib import Path

import soundfile as sf


def _corpus_source_meta():
    if CORPUS_KIND == "hf":
        name = _env("CORPUS_NAME") or _env("HF_DATASET")
        url = _env("CORPUS_URL") or f"https://huggingface.co/datasets/{_env('HF_DATASET')}"
    elif CORPUS_KIND == "ljspeech":
        name = _env("CORPUS_NAME") or Path(_env("LJSPEECH_DIR")).name
        url = _env("CORPUS_URL")
    else:
        name = _env("CORPUS_NAME") or Path(_env("CSV_PATH")).stem
        url = _env("CORPUS_URL")
    return name, url


CORPUS_NAME, CORPUS_URL = _corpus_source_meta()
HF_RESOLVED_REVISION = None  # filled in by _resolve_hf_revision() for CORPUS_KIND=hf


def _resolve_hf_revision():
    """The Hub commit sha actually read. Recorded in the .meta.json sidecar."""
    global HF_RESOLVED_REVISION
    from huggingface_hub import HfApi
    api = HfApi()
    info = api.dataset_info(_env("HF_DATASET"), revision=_env("HF_REVISION") or None)
    HF_RESOLVED_REVISION = info.sha
    return HF_RESOLVED_REVISION


def _iter_hf_rows(pace=False):
    import io
    from datasets import load_dataset
    from datasets.features import Audio

    dataset = _env("HF_DATASET")
    config = _env("HF_CONFIG") or None
    revision = HF_RESOLVED_REVISION or (_env("HF_REVISION") or None)
    text_field = _env("TEXT_FIELD")
    audio_field = _env("AUDIO_FIELD")
    speaker_field = _env("SPEAKER_FIELD")
    pace_every = _int_env("HF_PACE_EVERY", 50)
    pace_seconds = _float_env("HF_PACE_SECONDS", 1.0) or 0.0
    splits = [s.strip() for s in _env("HF_SPLITS").split(",") if s.strip()]

    for split in splits:
        ds = load_dataset(dataset, config, split=split, streaming=True, revision=revision)
        # decode=False: read each file's own bytes and decode them with soundfile below,
        # instead of the library's own audio decoder (which needs torch/torchcodec and
        # can resample). This is also how the notebook keeps every row at its native rate.
        ds = ds.cast_column(audio_field, Audio(decode=False))
        for i, row in enumerate(ds):
            if pace and pace_every and i > 0 and i % pace_every == 0 and pace_seconds:
                time.sleep(pace_seconds)
            key = f"{dataset}:{split}:{i}"
            audio_bytes = row[audio_field]["bytes"]
            assert audio_bytes is not None, (
                f"{key}: this dataset stores audio as an external path, not embedded bytes; "
                "add a reader branch for that case before using it here"
            )
            arr, sr = sf.read(io.BytesIO(audio_bytes), dtype="float32", always_2d=False)
            yield {
                "id": key,
                "text": row.get(text_field, "") or "",
                "speaker": row.get(speaker_field) if speaker_field else None,
                "audio_array": arr,
                "sample_rate": int(sr),
                "source": {"kind": "corpus", "name": CORPUS_NAME, "url": CORPUS_URL, "item": key},
            }


def _read_ljspeech_metadata_line(line):
    """id|text or id|text|normalized. The last field is the text of record."""
    parts = line.rstrip("\n").split("|")
    row_id = parts[0]
    text = parts[2] if len(parts) > 2 else parts[1]
    return row_id, text


def _iter_ljspeech_rows(pace=False):
    ljdir = Path(_env("LJSPEECH_DIR"))
    speaker = _env("LJSPEECH_SPEAKER") or None
    with open(ljdir / "metadata.csv", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            row_id, text = _read_ljspeech_metadata_line(line)
            key = f"{CORPUS_NAME}:{row_id}"
            yield {
                "id": key,
                "text": text,
                "speaker": speaker,
                "audio_path": str(ljdir / "wavs" / f"{row_id}.wav"),
                "source": {"kind": "corpus", "name": CORPUS_NAME, "url": CORPUS_URL, "item": row_id},
            }


def _iter_csv_rows(pace=False):
    csv_path = Path(_env("CSV_PATH"))
    base_dir = Path(_env("CSV_AUDIO_BASE_DIR") or csv_path.parent)
    text_col = _env("CSV_TEXT_COL")
    audio_col = _env("CSV_AUDIO_COL")
    speaker_col = _env("CSV_SPEAKER_COL") or None
    id_col = _env("CSV_ID_COL") or None
    with open(csv_path, encoding="utf-8") as f:
        for i, row in enumerate(csv.DictReader(f)):
            row_id = row[id_col] if id_col else str(i)
            key = f"{CORPUS_NAME}:{row_id}"
            yield {
                "id": key,
                "text": row.get(text_col, "") or "",
                "speaker": row.get(speaker_col) if speaker_col else None,
                "audio_path": str(base_dir / row[audio_col]),
                "source": {"kind": "corpus", "name": CORPUS_NAME, "url": CORPUS_URL, "item": row_id},
            }


def iter_corpus_rows(pace=False):
    if CORPUS_KIND == "hf":
        yield from _iter_hf_rows(pace=pace)
    elif CORPUS_KIND == "ljspeech":
        yield from _iter_ljspeech_rows(pace=pace)
    else:
        yield from _iter_csv_rows(pace=pace)


def _row_duration_s(row):
    """Cheap duration: a soundfile header read for a path, the array length for a decoded row."""
    if "audio_array" in row:
        return len(row["audio_array"]) / row["sample_rate"]
    info = sf.info(row["audio_path"])
    return info.frames / info.samplerate


def _load_audio_native(row):
    """The full audio at the corpus's own sample rate. This notebook never resamples."""
    if "audio_array" in row:
        return row["audio_array"], row["sample_rate"]
    arr, sr = sf.read(row["audio_path"], dtype="float32", always_2d=False)
    return arr, sr


if CORPUS_KIND == "hf":
    _resolve_hf_revision()
    print(f"Reading {_env('HF_DATASET')} at revision {HF_RESOLVED_REVISION}")


---
## 2 · Speaker accounting (first pass)

A first pass reads every row once to sum audio duration and count rows per speaker, without
keeping the audio in memory. Every distinct speaker value is reported with its row count; a value
that is empty or reads like a bucket (`unknown`, `other`, `multiple`) is named here and left out of
the `TOP_SPEAKERS` ranking below, since it does not identify one speaker.


In [ ]:
from collections import Counter

_BUCKET_SPEAKER_VALUES = {"", "unknown", "other", "multiple"}


def _speaker_key(row):
    sp = row["speaker"]
    return "" if sp is None else str(sp).strip()


def _is_bucket_speaker(sp):
    return sp.lower() in _BUCKET_SPEAKER_VALUES


speaker_rows = Counter()
speaker_seconds = Counter()
n_scanned = 0

for row in iter_corpus_rows(pace=True):
    n_scanned += 1
    sp = _speaker_key(row)
    speaker_rows[sp] += 1
    speaker_seconds[sp] += _row_duration_s(row)

print(f"Rows scanned: {n_scanned}")
print(f"{'speaker':30s} {'rows':>8} {'hours':>10}")
for sp, n in sorted(speaker_rows.items(), key=lambda kv: -kv[1]):
    label = sp if sp else "(empty)"
    print(f"{label:30s} {n:>8} {speaker_seconds[sp] / 3600:>10.4f}")

bucket_values = sorted(sp for sp in speaker_rows if _is_bucket_speaker(sp))
if bucket_values:
    print(f"\nExcluded from TOP_SPEAKERS (bucket values): {[v or '(empty)' for v in bucket_values]}")

ranked_speakers = sorted(
    ((sp, speaker_rows[sp], speaker_seconds[sp]) for sp in speaker_rows if not _is_bucket_speaker(sp)),
    key=lambda t: -t[2],
)
print(f"\nTop {min(20, len(ranked_speakers))} speakers by hours:")
print(f"{'speaker':30s} {'hours':>10} {'rows':>8}")
for sp, n, secs in ranked_speakers[:20]:
    print(f"{sp:30s} {secs / 3600:>10.4f} {n:>8}")


---
## 3 · Speaker selection

`SPEAKERS` names an explicit list of speakers to keep. `TOP_SPEAKERS` keeps the N speakers with
the most audio from the ranking above, bucket values excluded. Set at most one; leaving both empty
(or `TOP_SPEAKERS=0`) keeps every speaker — the right setting for a corpus that names none
(`ljspeech` with no `LJSPEECH_SPEAKER`, or `hf`/`csv` with no `SPEAKER_FIELD`/`CSV_SPEAKER_COL`).

| Variable | Meaning |
|---|---|
| `SPEAKERS` | Comma list of speaker ids to keep |
| `TOP_SPEAKERS` | Keep the N speakers with the most audio, 0 = off |


In [ ]:
_speakers_cfg = [s.strip() for s in _env("SPEAKERS").split(",") if s.strip()]
_top_n = _int_env("TOP_SPEAKERS", 0)

assert not (_speakers_cfg and _top_n), "set at most one of SPEAKERS or TOP_SPEAKERS"

if _speakers_cfg:
    keep_speakers = set(_speakers_cfg)
    speaker_rule = {"mode": "explicit_list", "speakers": sorted(keep_speakers)}
elif _top_n:
    keep_speakers = {sp for sp, _, _ in ranked_speakers[:_top_n]}
    speaker_rule = {"mode": "top_n", "n": _top_n, "speakers": sorted(keep_speakers)}
else:
    keep_speakers = None
    speaker_rule = {"mode": "all"}

print(f"Speaker rule: {speaker_rule}")
if keep_speakers is not None:
    kept_rows = sum(speaker_rows[sp] for sp in keep_speakers)
    print(f"Rows from kept speakers, before text validation: {kept_rows} of {n_scanned}")


---
## 4 · Text validation

A row is refused when its text is empty, contains a bracketed annotation (`[...]` or `<...>`), or
has no letter. Ingest only refuses a bad row; it never rewrites the text, and it computes no
quality score here — scoring is the measure notebook's job, not ingest's.


---
## 5 · Write the manifest

A second pass walks the corpus again, in the order it yields rows, and for each row that passes
the speaker rule and text validation: decodes its audio, asserts it is mono (a multi-channel file
is downmixed by averaging its channels), and writes it as PCM WAV at its own native sample rate.
This notebook never resamples; if it ever needed to, the row's `.meta.json` entry would name the
rate it converted from. Once the accumulated duration reaches `MAX_HOURS`, further rows are
counted as capped rather than written.

| Variable | Meaning |
|---|---|
| `MAX_HOURS` | Cap on the hours written, empty = no cap |
| `OUTPUT_DIR`, `MANIFEST_NAME` | Where the manifest, its sidecar and `wavs/` are written |


In [ ]:
_BRACKET_RE = re.compile(r"\[[^\]]*\]|<[^>]*>")


def _text_refusal_reason(text):
    if not text or not text.strip():
        return "empty"
    if _BRACKET_RE.search(text):
        return "bracketed_annotation"
    if not any(ch.isalpha() for ch in text):
        return "no_letter"
    return None


def _slug(s):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", s)


max_hours_cfg = _float_env("MAX_HOURS", None)
max_seconds = max_hours_cfg * 3600 if max_hours_cfg else None

out_dir = Path(OUTPUT_DIR)
wavs_dir = out_dir / "wavs"
wavs_dir.mkdir(parents=True, exist_ok=True)

reason_counts = Counter()
manifest_rows = []
downmixed_ids = []
n_seen = 0
n_speaker_filtered_out = 0
n_capped = 0
total_seconds = 0.0

for row in iter_corpus_rows(pace=False):
    n_seen += 1
    sp = _speaker_key(row)
    if keep_speakers is not None and sp not in keep_speakers:
        n_speaker_filtered_out += 1
        continue

    reason = _text_refusal_reason(row["text"])
    if reason:
        reason_counts[reason] += 1
        continue

    arr, sr = _load_audio_native(row)
    if arr.ndim > 1:
        arr = arr.mean(axis=1)
        downmixed_ids.append(row["id"])
    assert arr.ndim == 1, f"{row['id']}: audio is not mono after downmix"

    dur = len(arr) / sr
    if max_seconds is not None and total_seconds + dur > max_seconds:
        n_capped += 1
        continue
    total_seconds += dur

    out_name = f"{_slug(row['id'])}.wav"
    sf.write(str(wavs_dir / out_name), arr, sr, subtype="PCM_16")

    row_out = {
        "id": row["id"],
        "text": row["text"],
        "lang": CORPUS_LANG,
        "audio": f"wavs/{out_name}",
        "duration_s": round(dur, 3),
        "sample_rate": int(sr),
        "source": row["source"],
    }
    if sp:
        row_out["speaker"] = sp
    manifest_rows.append(row_out)
    reason_counts["passed"] += 1

print(f"Rows seen in the write pass  : {n_seen}")
print(f"Filtered out by speaker rule : {n_speaker_filtered_out}")
print("Text validation:")
for reason in ("empty", "bracketed_annotation", "no_letter", "passed"):
    print(f"  {reason:22s} {reason_counts.get(reason, 0)}")
print(f"Capped by MAX_HOURS          : {n_capped}")
print(f"Downmixed to mono            : {len(downmixed_ids)}")
print(f"Rows written                 : {len(manifest_rows)}  ({total_seconds / 3600:.4f} hours)")


---
## 6 · Manifest, sidecar and summary


In [ ]:
manifest_path = out_dir / _env("MANIFEST_NAME", "manifest.jsonl")
with open(manifest_path, "w", encoding="utf-8") as f:
    for row in manifest_rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

meta = {
    "notebooks": ["tts_corpus_ingest.ipynb"],
    "corpus": {
        "kind": CORPUS_KIND,
        "id": (
            _env("HF_DATASET") if CORPUS_KIND == "hf"
            else _env("LJSPEECH_DIR") if CORPUS_KIND == "ljspeech"
            else _env("CSV_PATH")
        ),
        "name": CORPUS_NAME,
        "config": (_env("HF_CONFIG") or None) if CORPUS_KIND == "hf" else None,
        "splits": ([s.strip() for s in _env("HF_SPLITS").split(",") if s.strip()]
                   if CORPUS_KIND == "hf" else None),
        "revision": HF_RESOLVED_REVISION if CORPUS_KIND == "hf" else None,
    },
    "speaker_rule": speaker_rule,
    "excluded_bucket_speaker_values": [v or "(empty)" for v in bucket_values],
    "counts": {
        "rows_scanned": n_scanned,
        "rows_speaker_filtered_out": n_speaker_filtered_out,
        "rows_text_refused": {k: v for k, v in reason_counts.items() if k != "passed"},
        "rows_capped_by_max_hours": n_capped,
        "rows_written": len(manifest_rows),
        "hours_written": round(total_seconds / 3600, 4),
        "max_hours_cap": max_hours_cfg,
        "downmixed_to_mono": len(downmixed_ids),
    },
}

meta_path = out_dir / (_env("MANIFEST_NAME", "manifest.jsonl") + ".meta.json")
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print(f"Manifest : {manifest_path}  ({len(manifest_rows)} rows)")
print(f"Sidecar  : {meta_path}")
print(f"Audio    : {wavs_dir}")
if manifest_rows:
    print("First row:", json.dumps(manifest_rows[0], ensure_ascii=False))


---
Licensed under Apache-2.0.
